# South Sudan In-situ Data Layers
## Data Access

The data is stored in the following Google Cloud Storage bucket (source from GMV):
- https://console.cloud.google.com/storage/browser/wbhydross_deliverables

**Input Data**

Hurst Nile Basin Volumes (HYDROC) data:
- Vol1: `wbhydross_deliverables/D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol1`
- Vol2: `wbhydross_deliverables/D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol2`
- Vol3: `wbhydross_deliverables/D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol3`
Ministry of Water Resources and Irrigation (MWRI) Data:
- `wbhydross_deliverables/D3-Database/012-Data rescue/MWRI/MWRI_river_discharge_json`


## Setup

### Library import

In [8]:
from urllib.parse import quote

import geopandas as gpd
import pandas as pd
import requests
from google.cloud import storage
from shapely.geometry import Point

## Create layers

In [22]:
folder_paths = {
    "MWRI": "D3-Database/012-Data rescue/MWRI/MWRI_river_discharge_json/",
    "HYDROC_Vol1": "D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol1",
    "HYDROC_Vol2": "D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol2",
    "HYDROC_Vol3": "D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol3",
}

file_names = {
    "MWRI": "MWRI_processed_stations.shp",
    "HYDROC_Vol1": "HYDROC_stations_Vol1.shp",
    "HYDROC_Vol2": "HYDROC_stations_Vol2.shp",
    "HYDROC_Vol3": "HYDROC_stations_Vol3.shp",
}

volumes = {
    "MWRI": None,
    "HYDROC_Vol1": 1,
    "HYDROC_Vol2": 2,
    "HYDROC_Vol3": 3,
}

In [26]:
client = storage.Client(project="wb-hydro-ss")

bucket_name = "wbhydross_deliverables"

for name, folder_path in folder_paths.items():
    # List all the blobs in the specified folder
    bucket = client.get_bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=folder_path)

    # Filter the blobs to get only the .json files
    json_files = [blob for blob in blobs if blob.name.endswith(".json")]

    data = []
    for json_file in json_files:
        public_url = quote(
            f"https://storage.googleapis.com/{bucket_name}/{json_file.name}", safe=":/"
        )

        # Fetch the content from the public URL
        response = requests.get(public_url)
        content = response.json()
        data.append(content["info"])

    # Convert the list of dictionaries to a pandas DataFrame
    df = pd.DataFrame(data)
    df["Vol"] = volumes[name]

    # Create a GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df["Lon"], df["Lat"])])
    gdf.drop(columns=["Lat", "Lon"], inplace=True)
    gdf.rename(columns={"Creation date": "Date", "Data Quality": "Quality"}, inplace=True)

    # Set the coordinate reference system (CRS) to EPSG:4326
    gdf = gdf.set_crs(epsg=4326)

    # Save the GeoDataFrame to a shapefile
    gdf.to_file(f"../data/processed/In-situ-data/{file_names[name]}")